# 01 · Live data sources

**Goal:** find out which model inputs the product can fetch live, every hour, for free.
Only features available at serving time can be used, otherwise the model trains on data it will never see live (train/serve skew).

## 1. Setup

In [1]:
# Cell 3: Setup.
import os                                               # Read the API key from the environment
from datetime import datetime, timedelta, timezone      # Time windows for the API queries
from pathlib import Path                                # Find the project folder
import httpx                                            # HTTP client for the APIs
import pandas as pd                                     # Tables
from dotenv import load_dotenv                          # Reads KEY=value lines from .env into the environment

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # Project folder, wherever it lives
load_dotenv(ROOT / ".env")                              # Load the secrets file (it is never committed to git)
OPENAQ_KEY = os.environ.get("OPENAQ_API_KEY", "")
SHADIPUR = (28.6517, 77.1582)                           # Shadipur CPCB station (latitude, longitude)
WEATHER_POINT = (28.646748, 77.17218)                   # The Open-Meteo grid point P1 trained on (from weather.csv)
print("OpenAQ key set:", bool(OPENAQ_KEY))              # Must be True (prints only True/False, never the key)

OpenAQ key set: True


## 2. OpenAQ: monitoring stations near Shadipur
Which stations are within 3 km, who provides them, which pollutants they measure, and when they last reported.

In [2]:
# Cell 5: Find OpenAQ locations within 3 km of Shadipur.
API = "https://api.openaq.org/v3"
HEADERS = {"X-API-Key": OPENAQ_KEY}                     # OpenAQ requires the key in this header
r = httpx.get(f"{API}/locations", headers=HEADERS, timeout=30,
              params={"coordinates": f"{SHADIPUR[0]},{SHADIPUR[1]}", "radius": 3000, "limit": 20})
r.raise_for_status()                                    # Stop with a clear error if the request failed
locations = r.json()["results"]

for loc in locations:                                   # One block per station
    last = (loc.get("datetimeLast") or {}).get("local", "?")
    print(f"[{loc['id']}] {loc['name']} | provider: {(loc.get('provider') or {}).get('name')} | last data: {last}")
    for s in loc.get("sensors", []):                    # The pollutants this station measures
        print(f"      sensor {s['id']}: {s['parameter']['name']} ({s['parameter']['units']})")

[2503] Shadipur, Delhi - CPCB | provider: CPCB | last data: 2018-02-22T08:15:00+05:30
      sensor 4851: co (µg/m³)
      sensor 4852: no2 (µg/m³)
      sensor 4853: o3 (µg/m³)
      sensor 13860: pm25 (µg/m³)
      sensor 13859: so2 (µg/m³)
[5404] Pusa, Delhi - IMD | provider: CPCB | last data: 2026-11-16T13:30:00+05:30
      sensor 14086: co (µg/m³)
      sensor 12234697: co (ppb)
      sensor 12234698: no (ppb)
      sensor 12234699: no2 (ppb)
      sensor 14084: no2 (µg/m³)
      sensor 14340700: nox (ppb)
      sensor 12234700: o3 (µg/m³)
      sensor 12234701: pm10 (µg/m³)
      sensor 14085: pm10 (µg/m³)
      sensor 12234702: pm25 (µg/m³)
      sensor 14087: pm25 (µg/m³)
[5610] North Campus, DU, Delhi - IMD | provider: CPCB | last data: 2026-11-16T10:45:00+05:30
      sensor 14909: co (µg/m³)
      sensor 12234685: co (ppb)
      sensor 12234686: no (ppb)
      sensor 14902: no2 (µg/m³)
      sensor 12234687: no2 (ppb)
      sensor 14340698: nox (ppb)
      sensor 14889: o3 (µg

## 3. OpenAQ: the last 72 hours of PM2.5 at Shadipur
Is the data live (updated within the last few hours)? Are there gaps? The model needs 48 hours of history.

In [3]:
# Cell 7: Pick the live Shadipur station, then check every one of its sensors over the last 72 hours.
candidates = [l for l in locations if "shadipur" in l["name"].lower()]
shadipur = max(candidates, key=lambda l: (l.get("datetimeLast") or {}).get("utc", ""))   # The most recently active one
print("Using:", shadipur["id"], shadipur["name"], "| last data:", shadipur["datetimeLast"]["local"])


def sensor_hours(sensor_id, hours=72):
    """Hourly values of one sensor for the last `hours` hours, indexed by local (Indian) time."""
    now = datetime.now(timezone.utc)
    r = httpx.get(f"{API}/sensors/{sensor_id}/hours", headers=HEADERS, timeout=30,
                  params={"datetime_from": (now - timedelta(hours=hours)).isoformat(),
                          "datetime_to": now.isoformat(), "limit": 200})
    r.raise_for_status()
    results = r.json()["results"]
    return pd.Series({pd.Timestamp(h["period"]["datetimeFrom"]["local"][:19]): h["value"] for h in results},
                     dtype=float).sort_index()   # [:19] drops the "+05:30" so times match the weather data


series, summary = {}, []
for s in shadipur["sensors"]:                           # Every sensor at the station
    data = sensor_hours(s["id"])
    series[s["id"]] = data
    summary.append({"sensor": s["id"], "parameter": f"{s['parameter']['name']} ({s['parameter']['units']})",
                    "hours (of 72)": len(data), "latest": data.index.max() if len(data) else None,
                    "mean": round(data.mean(), 1) if len(data) else None})

pm25_ids = [s["id"] for s in shadipur["sensors"] if s["parameter"]["name"] == "pm25"]
best = max(pm25_ids, key=lambda i: len(series[i]))      # The PM2.5 sensor with the most data
station = series[best].rename("pm25").to_frame()        # Used by Cell 11
print(f"PM2.5 sensor used: {best} | hours: {len(station)} | latest: {station.index.max()} | now: {datetime.now():%Y-%m-%d %H:%M}")
pd.DataFrame(summary).sort_values("parameter")

Using: 5630 Shadipur, Delhi - CPCB | last data: 2026-11-16T14:00:00+05:30
PM2.5 sensor used: 12234726 | hours: 71 | latest: 2026-11-16 13:00:00 | now: 2026-11-16 15:35


,sensor,parameter,hours (of 72),latest,mean
0,12234721,co (ppb),71,2026-11-16 13:00:00,0.8
1,15119,co (µg/m³),0,NaT,NaN
2,12234722,no (ppb),71,2026-11-16 13:00:00,15.3
4,12234723,no2 (ppb),71,2026-11-16 13:00:00,125.1
3,15053,no2 (µg/m³),0,NaT,NaN
5,14341721,nox (ppb),71,2026-11-16 13:00:00,0.1
6,12234724,o3 (µg/m³),71,2026-11-16 13:00:00,5.8
7,15011,o3 (µg/m³),0,NaT,NaN
9,12234725,pm10 (µg/m³),71,2026-11-16 13:00:00,259.0
8,36210,pm10 (µg/m³),0,NaT,NaN


## 4. Open-Meteo: tomorrow's weather (same variables as P1's training data)

In [4]:
# Cell 9: Weather for the past 2 days and the next 2 days, hourly, in Indian time.
WEATHER_VARS = ["temperature_2m", "relative_humidity_2m", "dew_point_2m", "precipitation",
                "wind_speed_10m", "wind_direction_10m", "surface_pressure", "cloud_cover"]   # Exactly P1's columns
r = httpx.get("https://api.open-meteo.com/v1/forecast", timeout=30, params={
    "latitude": WEATHER_POINT[0], "longitude": WEATHER_POINT[1], "hourly": ",".join(WEATHER_VARS),
    "past_days": 2, "forecast_days": 2, "timezone": "Asia/Kolkata"})   # km/h wind and hPa pressure by default, as in P1
r.raise_for_status()
weather = pd.DataFrame(r.json()["hourly"]).set_index("time")
weather.index = pd.to_datetime(weather.index)
print(f"Weather hours: {len(weather)} | from {weather.index.min()} to {weather.index.max()}")
weather.tail(3)

Weather hours: 96 | from 2026-11-14 00:00:00 to 2026-11-17 23:00:00


,temperature_2m,relative_humidity_2m,dew_point_2m,precipitation,wind_speed_10m,wind_direction_10m,surface_pressure,cloud_cover
time,,,,,,,,
2026-11-17 21:00:00,18.1,65,11.5,0.0,4.9,264,989.0,0
2026-11-17 22:00:00,17.5,67,11.3,0.0,4.4,251,989.3,0
2026-11-17 23:00:00,16.9,69,11.2,0.0,3.3,257,989.6,0


## 5. Open-Meteo Air Quality (CAMS): model-estimated PM2.5
Not a measurement but a global atmospheric model. Compare it with the station: is it close enough to be a fallback?
Its forecast for tomorrow is also a competitor our model should beat.

In [5]:
# Cell 11: CAMS PM2.5 for the past 3 days and the next 2, and a comparison with the station.
r = httpx.get("https://air-quality-api.open-meteo.com/v1/air-quality", timeout=30, params={
    "latitude": SHADIPUR[0], "longitude": SHADIPUR[1], "hourly": "pm2_5,pm10",
    "past_days": 3, "forecast_days": 2, "timezone": "Asia/Kolkata"})
r.raise_for_status()
cams = pd.DataFrame(r.json()["hourly"]).set_index("time")
cams.index = pd.to_datetime(cams.index)

both = station.join(cams["pm2_5"].rename("cams_pm25"), how="inner").dropna()   # Hours where both have a value
print(f"Overlapping hours: {len(both)}")
print(f"Station mean: {both['pm25'].mean():.0f} | CAMS mean: {both['cams_pm25'].mean():.0f} µg/m³")
print(f"Correlation: {both['pm25'].corr(both['cams_pm25']):.2f}")
both.tail(10)

Overlapping hours: 71
Station mean: 153 | CAMS mean: 89 µg/m³
Correlation: 0.45


,pm25,cams_pm25
2026-11-16 04:00:00,183.0,105.7
2026-11-16 05:00:00,233.0,91.3
2026-11-16 06:00:00,326.0,91.7
2026-11-16 07:00:00,312.0,92.2
2026-11-16 08:00:00,231.0,94.1
2026-11-16 09:00:00,143.0,91.7
2026-11-16 10:00:00,135.0,82.9
2026-11-16 11:00:00,97.2,70.1
2026-11-16 12:00:00,87.6,67.3
2026-11-16 13:00:00,81.9,68.2
